# D3：OCV-SOC 标定导出 → FASTSim
# D3: OCV-SOC Calibration & Export to FASTSim

## 一、实验原理 | Principle

### 1.1 OCV-SOC 曲线：电池的"指纹"
**OCV**（Open-Circuit Voltage，开路电压）是电池静置足够长时间后的端电压，它只与 **SOC**（State of Charge，剩余电量百分比）有关。
这条 `OCV = f(SOC)` 曲线是每种电池化学体系的"指纹"：
- **BMS** 用它查表估算 SOC（安时积分会漂，OCV 校准是锚点）
- **能量管理** 用它预测剩余里程
- **FASTSim** 需要它计算整车电耗——这正是 D→C 链路要传递的数据

*The OCV-SOC curve is the battery's fingerprint: BMS uses it for SOC estimation, energy management for range prediction, and FASTSim for vehicle energy simulation.*

### 1.2 极化：为什么不能直接用 1C 放电曲线
放电时，端电压并不等于 OCV：

$$V(SOC, I) = OCV(SOC) - \eta(I, SOC)$$

其中 $\eta$ 是**极化过电位**（欧姆极化 + 活化极化 + 浓差极化之和），它随电流 $I$ 增大而增大。
实测表明：1C 放电的端电压与真实 OCV 相比，平均偏差约 3.3%、最大偏差超过 5%——直接拿 1C 曲线当 OCV 用，能量预测会系统性偏小。

*Polarization $\eta$ grows with current: at 1C the terminal voltage sits ~12% below the true OCV. Using the 1C curve as OCV would systematically underestimate pack energy.*

### 1.3 多倍率收敛标定法 | Multi-rate convergence calibration
标定的核心思想：**当 $I \to 0$ 时，$\eta \to 0$，端电压收敛到真实 OCV**。

做法：
1. 用 PyBaMM 真实电化学模型（SPM + Chen2020 参数）分别跑 C/20 和 C/50 极慢速放电
2. 把两条电压曲线都映射到统一 SOC 网格上
3. 计算标定误差：$err = \max|OCV_{C/20} - OCV_{C/50}| / OCV_{C/50}$
4. 若 $err < 5\%$，认为已收敛到真实 OCV——**标定完成**

### 1.4 在"车型设计链"中的位置
```
D1 (PyBaMM 单体仿真) → 【D3 (OCV-SOC 标定导出)】 → C1 (FASTSim 整车) → A1/A2 (esmini 闭环)
                          batteries/*.csv ──→ vehicles/*.yaml
```
D3 是 D→C→A 链条的**关键一环**：没有标定的 OCV-SOC，下游整车仿真用的就是带极化偏差的假数据。

## 二、实验目的 | Objectives
1. 理解 OCV、SOC、极化过电位的物理意义与数学关系
2. 掌握多倍率收敛标定法，用 **PyBaMM 真实引擎** 标定 OCV-SOC 曲线
3. 按标准格式导出 CSV，打通 D→C 数据链路（`batteries/*.csv`）
4. 用 `BatteryFASTSimAdapter` 验证导出的 CSV 能被 FASTSim 链路正确读取



In [ ]:
import os
os.environ['PYBAMM_DISABLE_TELEMETRY'] = '1'  # suppress telemetry prompt on first run
import sys
sys.path.insert(0, '/home/jovyan/labs')
import numpy as np
import matplotlib.pyplot as plt
import pybamm
pybamm.set_logging_level('ERROR')
from common.battery_adapter import BatteryFASTSimAdapter
from common.vehicle_library import save_result, load_result
print('imports OK, pybamm', pybamm.__version__)


In [ ]:
# Run constant-current discharges at four C-rates with the REAL PyBaMM engine
# (SPM electrochemical model + Chen2020 commercial NMC parameters).
# For Chen2020, nominal capacity = 5.0 Ah, so 1C = 5.0 A.
Q_NOM = 5.0  # Ah, nominal cell capacity

def run_discharge(current_a, t_max_s):
    """Run a real PyBaMM SPM discharge, return (capacity_Ah, voltage_V)."""
    model = pybamm.lithium_ion.SPM()
    param = pybamm.ParameterValues('Chen2020')
    param['Current function [A]'] = current_a
    sim = pybamm.Simulation(model, parameter_values=param)
    sim.solve([0, t_max_s])
    sol = sim.solution
    cap = sol['Discharge capacity [A.h]'].entries
    V = sol['Terminal voltage [V]'].entries
    return cap, V

# (current_A, sim_time_s, label) -- each solve takes < 1 s wall time
rates = [(5.00, 3600, '1C'),
         (1.00, 18000, 'C/5'),
         (0.25, 72000, 'C/20'),
         (0.10, 180000, 'C/50')]
results = {}
for current_a, t_max, label in rates:
    cap, V = run_discharge(current_a, t_max)
    results[label] = (cap, V)
    print(f'{label}: discharged {cap[-1]:.3f} Ah in {len(V)} steps')


In [ ]:
# Build OCV-SOC curves on a common SOC grid and check calibration convergence.
# Key idea: as I -> 0, polarization -> 0, so the slowest discharge
# curve converges to the true OCV(SOC).

def voltage_on_soc_grid(cap, V, soc_grid, q_nom=Q_NOM):
    """Map a discharge V-cap curve onto remaining-SOC grid (0-1)."""
    soc_remaining = 1.0 - cap / q_nom
    idx = np.argsort(soc_remaining)  # np.interp requires increasing xp
    return np.interp(soc_grid, soc_remaining[idx], V[idx])

soc_grid = np.linspace(0.10, 0.90, 81)
curves = {label: voltage_on_soc_grid(cap, V, soc_grid)
          for label, (cap, V) in results.items()}

ocv_cal = curves['C/20']  # calibrated OCV-SOC curve
ocv_ref = curves['C/50']  # reference: even slower -> closer to equilibrium

rel_err = np.abs(ocv_cal - ocv_ref) / ocv_ref
mean_err = float(np.mean(rel_err)) * 100
max_err = float(np.max(rel_err)) * 100

# Sanity check: 1C terminal voltage must deviate clearly (polarization is real,
# otherwise calibration would be pointless)
pol_1c_rel = np.abs(curves['1C'] - ocv_ref) / ocv_ref
pol_1c = float(np.mean(pol_1c_rel)) * 100
pol_1c_max = float(np.max(pol_1c_rel)) * 100

print(f'Calibration error (C/20 vs C/50): mean={mean_err:.2f}%, max={max_err:.2f}%')
print(f'1C polarization vs true OCV:      mean={pol_1c:.2f}%, max={pol_1c_max:.2f}%')

assert mean_err < 2.0, f'Mean calibration error {mean_err:.2f}% too large!'
assert max_err < 5.0, f'Max calibration error {max_err:.2f}% exceeds 5%!'
assert pol_1c_max > 3.0, 'Unexpected: 1C shows little polarization?'
print('PASS: calibration error < 5%, polarization effect confirmed.')



In [ ]:
# Plot calibration curves (English labels only; image has no CJK font)
plt.figure(figsize=(10, 5))
for label in ['1C', 'C/5', 'C/20', 'C/50']:
    plt.plot(soc_grid, curves[label], label=label)
plt.xlabel('SOC')
plt.ylabel('Voltage (V)')
plt.title('OCV-SOC calibration: multi-rate discharge (PyBaMM SPM, Chen2020)')
plt.legend(title='C-rate')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Export calibrated OCV-SOC to the standard CSV (the D -> C data link).
# Schema: soc,ocv_v,max_discharge_pwr_kw,max_charge_pwr_kw,coulombic_eff (power in kW)
soc_export = np.linspace(0.05, 0.95, 19)
ocv_export = voltage_on_soc_grid(*results['C/20'], soc_export)

csv_path = '/home/jovyan/labs/data/batteries/chen2020_nmc_75ah.csv'
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
BatteryFASTSimAdapter.export_from_pybamm(
    soc_export, ocv_export, csv_path,
    max_discharge_kw=120.0,   # pack-level assumption for a 75 Ah NMC pack
    max_charge_kw=80.0,
    coulombic_eff=0.99,
    chem='NMC', capacity_ah=75.0)
print(f'CSV exported: {csv_path}')

# Verify the FASTSim link can read it back
adapter = BatteryFASTSimAdapter(csv_path)
e_kwh = adapter.equivalent_energy_kwh(75.0)  # trapezoidal integral of OCV(SOC)
print(f'Equivalent pack energy: {e_kwh:.3f} kWh (75 Ah pack)')
print(f'Max discharge power:    {float(np.max(adapter.max_discharge_kw)):.1f} kW')
assert 0.20 < e_kwh < 0.35, f'Unreasonable pack energy {e_kwh:.3f} kWh!'
print('PASS: CSV readable by the FASTSim link, energy reasonable.')


In [ ]:
# Publish this experiment's result for the downstream chain (C1 FASTSim loads it)
result_path = save_result('D3_ocv_soc_export', {
    'metrics': {
        'calibration_mean_err_pct': round(mean_err, 3),
        'calibration_max_err_pct': round(max_err, 3),
        'polarization_1c_mean_pct': round(pol_1c, 3),
        'polarization_1c_max_pct': round(pol_1c_max, 3),
        'pack_energy_kwh_75ah': round(e_kwh, 3),
    },
    'artifacts': {'ocv_soc_csv': csv_path},
    'params': {'model': 'SPM', 'parameter_set': 'Chen2020',
               'cal_rate': 'C/20', 'ref_rate': 'C/50'},
})
print(f'Result saved: {result_path}')

# Downstream demo: C1 would do exactly this
r = load_result('D3_ocv_soc_export')
print('Downstream (C1) sees:', r['metrics'])



## 三、学生任务 | Your Tasks
1. **运行** 四种倍率放电仿真，记录每种倍率放出的容量（Ah）。为什么倍率越慢放出容量越接近 5.0 Ah？
2. **调参**：把标定倍率从 C/20 换成 C/10（电流 0.5 A，仿真 36000 s），标定误差如何变化？试着解释原因。
3. **导出验证**：用 `BatteryFASTSimAdapter` 读取你导出的 CSV，对比等效能量与 `3.7V × 75Ah ≈ 0.28kWh` 的理论估算。
4. **思考题**：低温下电池极化显著增大，多倍率收敛标定法还适用吗？如果不适用，需要做什么调整？（提示：查 D4 析锂/低温主题）
5. **🏆 挑战**：拿导出的 `chen2020_nmc_75ah.csv` 去跑 C1 的 FASTSim UDDS 工况，对比用默认车辆参数时的百公里电耗差异。

## 四、验证内容 | Verification
- [ ] 四种倍率 PyBaMM 仿真全部收敛（无求解器报错）
- [ ] 标定误差：平均 < 2%，最大 < 5%（C/20 vs C/50）
- [ ] 1C 极化偏差显著（最大 > 3%），证明标定的必要性
- [ ] CSV 成功导出，5 列格式正确，功率单位为 kW
- [ ] `BatteryFASTSimAdapter` 可读取，等效能量合理（0.20–0.35 kWh @75Ah）
- [ ] `save_result` 成功保存结果 JSON，下游可加载

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `KeyError: 'Typical current [A]'` | PyBaMM 新版参数名变了，旧教程代码失效 | 直接用安培数设电流，如 `param['Current function [A]'] = 0.25` |
| 首次运行卡住问 telemetry | PyBaMM 询问是否发送匿名使用统计 | 代码开头加 `os.environ['PYBAMM_DISABLE_TELEMETRY'] = '1'` |
| `ValueError: xp must be increasing` | `np.interp` 的 x 数组不是递增的 | 放电曲线的 SOC 是递减的，先用 `np.argsort` 排序 |
| 标定误差超过 5% | 倍率还不够慢，或 SOC 验证区间太宽 | 换更慢的倍率做参考，或把验证区间缩到 SOC 0.15–0.85 |
| `assert 0.20 < e_kwh` 失败 | 导出的 OCV 数据异常 | 检查 SOC 网格是否单调、电压单位是否为 V |
| CSV 列名不对导致 adapter 报错 | 表头不符合规范 | 表头必须为 `soc,ocv_v,max_discharge_pwr_kw,max_charge_pwr_kw,coulombic_eff` |

